# Test Case Selection Exercise

In this exercise, we'll practice **test case selection** — given a large suite for `calculate_shipping_fee` and a *changed* version of the function, automatically pick only the tests that are **behaviorally relevant** to the change.

We'll use a **coverage-based** safe selection (Rothermel & Harrold, 1994 — see [the regression-testing slide deck](../1_regression_testing.md)):
- Identify the **changed statements** between the old and new versions.
- Select every test whose execution on the **old** version touched at least one changed statement.

We'll walk through two change scenarios:
- **Scenario A (simple)**: shift the free-shipping threshold (`order_total < 40000` → `< 50000`).
- **Scenario B (compound)**: tighten the WOW discount (`// 2` → `// 3`) and the NEW_USER coupon (`-= 2000` → `-= 3000`).

## Software under Test (SuT)

We define the **original** `calculate_shipping_fee` and the two **changed** versions directly in cells below. To keep statement-level diffs clean, the three versions share the *exact same line structure* — only the contents of the changed lines differ. Lines marked with a trailing `# CHANGED` comment indicate what each scenario modifies.

In [41]:
def calculate_shipping_fee(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    """Calculate shipping fee based on order and delivery conditions."""
    fee = 0

    # 1. base fee
    if order_total < 40000:
        fee += 3000

    # 2. weight surcharge
    if weight_kg <= 5:
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000

    # 3. distance surcharge
    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    # 4. island surcharge
    if is_island:
        fee += 4000

    # 5. membership discount
    if membership == "WOW":
        fee = fee // 2

    # 6. coupon discount
    if coupon_type == "NEW_USER":
        fee -= 2000

    # 7. final lower bound
    return max(fee, 0)

### Scenario A — free-shipping threshold change (single line)

In [42]:
def calculate_shipping_fee_A(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    """Calculate shipping fee based on order and delivery conditions."""
    fee = 0

    # 1. base fee
    if order_total < 50000:                  # CHANGED: 40000 -> 50000
        fee += 3000

    # 2. weight surcharge
    if weight_kg <= 5:
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000

    # 3. distance surcharge
    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    # 4. island surcharge
    if is_island:
        fee += 4000

    # 5. membership discount
    if membership == "WOW":
        fee = fee // 2

    # 6. coupon discount
    if coupon_type == "NEW_USER":
        fee -= 2000

    # 7. final lower bound
    return max(fee, 0)

### Scenario B — WOW discount and NEW_USER coupon change (two lines)

In [43]:
def calculate_shipping_fee_B(
    order_total: int,
    weight_kg: float,
    distance_km: int,
    is_island: bool,
    membership: str,
    coupon_type: str,
) -> int:
    """Calculate shipping fee based on order and delivery conditions."""
    fee = 0

    # 1. base fee
    if order_total < 40000:
        fee += 3000

    # 2. weight surcharge
    if weight_kg <= 5:
        fee += 0
    elif weight_kg <= 20:
        fee += 2000
    else:
        fee += 5000

    # 3. distance surcharge
    if distance_km <= 10:
        fee += 0
    elif distance_km <= 50:
        fee += 1000
    else:
        fee += 3000

    # 4. island surcharge
    if is_island:
        fee += 4000

    # 5. membership discount
    if membership == "WOW":
        fee = fee // 3                       # CHANGED: // 2 -> // 3

    # 6. coupon discount
    if coupon_type == "NEW_USER":
        fee -= 3000                          # CHANGED: 2000 -> 3000

    # 7. final lower bound
    return max(fee, 0)

In [44]:
# Sanity check: same input, different outputs across versions
inp = (30000, 10.0, 20, False, "WOW", "NEW_USER")
print(f"original    : {calculate_shipping_fee(*inp)}")
print(f"scenario A  : {calculate_shipping_fee_A(*inp)}")
print(f"scenario B  : {calculate_shipping_fee_B(*inp)}")

original    : 1000
scenario A  : 1000
scenario B  : 0


## Build a Large Test Suite

Same category-grid suite as the **test suite minimization** exercise. Each test is `(input_tuple, expected_output)` with the expected output taken from the **original** function — these are the assertions a regression test would check after the change.

In [45]:
from itertools import product

ORDER_TOTAL_VALUES = [10000, 39999, 45000, 50000]
WEIGHT_VALUES      = [3.0, 5.0, 20.0, 25.0]
DISTANCE_VALUES    = [5, 10, 50, 100]
IS_ISLAND_VALUES   = [True, False]
MEMBERSHIP_VALUES  = ["WOW", "NONE"]
COUPON_VALUES      = ["NEW_USER", "NONE"]

def build_test_suite():
    suite = []
    for inp in product(
        ORDER_TOTAL_VALUES, WEIGHT_VALUES, DISTANCE_VALUES,
        IS_ISLAND_VALUES, MEMBERSHIP_VALUES, COUPON_VALUES,
    ):
        expected = calculate_shipping_fee(*inp)
        suite.append((inp, expected))
    return suite

test_suite = build_test_suite()
print(f"Generated {len(test_suite)} tests")

Generated 512 tests


## Statement-Level Instrumentation (via `sys.settrace`)

For each test we need the set of **line numbers** executed in the original function. We use Python's built-in `sys.settrace` — the same mechanism real coverage tools (e.g., `coverage.py`) use under the hood.

We record line numbers **relative to the function's `def` line** (so `def calculate_shipping_fee(` is line 1, the first body line is line 2, etc.). That lets us compare line numbers consistently across the original and the changed versions, even though they live in different cells.

In [46]:
import sys

def trace_executed_lines(func, *args, **kwargs):
    """Run `func(*args, **kwargs)` and return (executed_relative_line_numbers, result).

    Line numbers are RELATIVE to the function's own `def` line (which is 1).
    """
    executed = set()
    target_code = func.__code__
    base = target_code.co_firstlineno - 1  # subtract to convert absolute -> relative

    def line_tracer(frame, event, arg):
        if event == "line":
            executed.add(frame.f_lineno - base)
        return line_tracer

    def call_tracer(frame, event, arg):
        if event == "call" and frame.f_code is target_code:
            return line_tracer
        return None

    sys.settrace(call_tracer)
    try:
        result = func(*args, **kwargs)
    finally:
        sys.settrace(None)
    return executed, result

# Demo: trace a single test
demo_inp = (10000, 3.0, 5, False, "WOW", "NEW_USER")
lines, result = trace_executed_lines(calculate_shipping_fee, *demo_inp)
print(f"Test input: {demo_inp}")
print(f"Executed lines (relative to def): {sorted(lines)}")
print(f"Result: {result}")

Test input: (10000, 3.0, 5, False, 'WOW', 'NEW_USER')
Executed lines (relative to def): [10, 13, 14, 17, 18, 25, 26, 33, 37, 38, 41, 42, 45]
Result: 0


### Collect per-test coverage

For every test we record the set of executed (relative) lines in the original SuT.

In [47]:
test_coverages = []
for inp, _expected in test_suite:
    lines, _ = trace_executed_lines(calculate_shipping_fee, *inp)
    test_coverages.append(lines)

print(f"Collected coverage for {len(test_coverages)} tests")
print(f"Coverage of test 0: {sorted(test_coverages[0])}")
print(f"Coverage of test 1: {sorted(test_coverages[1])}")
print(f"Coverage of test 2: {sorted(test_coverages[2])}")
print(f"Coverage of test 3: {sorted(test_coverages[3])}")

Collected coverage for 512 tests
Coverage of test 0: [10, 13, 14, 17, 18, 25, 26, 33, 34, 37, 38, 41, 42, 45]
Coverage of test 1: [10, 13, 14, 17, 18, 25, 26, 33, 34, 37, 38, 41, 45]
Coverage of test 2: [10, 13, 14, 17, 18, 25, 26, 33, 34, 37, 41, 42, 45]
Coverage of test 3: [10, 13, 14, 17, 18, 25, 26, 33, 34, 37, 41, 45]


## Identify Changed Statements

A changed statement is any line whose source text differs between the old and new versions. We get each function's source with `inspect.getsource()` and diff them line by line.

Since our scenarios only replace text in place (no lines added/removed) AND the three versions share the same line structure, a simple zip-and-compare suffices.

In [48]:
import inspect, textwrap

def diff_changed_lines(old_func, new_func) -> set:
    """Return the (function-relative) line numbers whose source text differs.

    Note on the `def` line: since our three versions live in separate cells
    they must use different function names (`calculate_shipping_fee`,
    `..._A`, `..._B`). That makes line 1 (the `def` line) differ in source
    text, even though no behavior changed there. Such header-only
    differences would never show up in trace coverage anyway, but we
    explicitly filter them out for a cleaner report.
    """
    old = textwrap.dedent(inspect.getsource(old_func)).splitlines()
    new = textwrap.dedent(inspect.getsource(new_func)).splitlines()
    assert len(old) == len(new), "scenarios assume line-aligned in-place edits"
    changed = {i + 1 for i, (a, b) in enumerate(zip(old, new)) if a != b}
    # filter out def-header lines (line 1 is always the function name line)
    return {ln for ln in changed if not old[ln - 1].lstrip().startswith("def ")}

changed_lines_A = diff_changed_lines(calculate_shipping_fee, calculate_shipping_fee_A)
changed_lines_B = diff_changed_lines(calculate_shipping_fee, calculate_shipping_fee_B)
print(f"Scenario A changed lines: {sorted(changed_lines_A)}")
print(f"Scenario B changed lines: {sorted(changed_lines_B)}")


Scenario A changed lines: [13]
Scenario B changed lines: [38, 42]


## Selection + Safety Check

**Selection** rule (safe, change-impact): keep every test whose old-version coverage intersects the changed lines.

**Safety check**: a test that does **not** touch any changed code must produce the same output on the new version (the executed code is identical). So the set of tests that *actually* fail on the new version (output differs from the original) must be a subset of the selected tests. We verify this for each scenario.

In [49]:
def select_tests(test_coverages, changed_lines):
    """Pick the test ids whose old-version coverage touches any changed line.

    `test_coverages[tid]` is a set of line numbers the test executed on the
    OLD function. A test is selected iff that set overlaps `changed_lines`.
    """
    selected = []                                  # tids that survive selection
    for tid, cov in enumerate(test_coverages):     # iterate over every test
        overlap = cov & changed_lines              # lines this test touched AND that changed
        if overlap:                                # any overlap -> the test may behave differently now
            selected.append(tid)                   # keep it
    return selected


def regression_failures(test_suite, new_func):
    """Tests whose output on the NEW function differs from the recorded `expected`.

    Terminology note: "fail" here follows standard regression-testing usage
    and does NOT mean the test itself is broken. It just means the
    assertion `actual == expected` no longer holds, because the new SuT
    behaves differently from the old one on this input. That difference
    could be:
      - a real regression (an unintended bug introduced by the change), or
      - an intentional behavioral change (the spec moved, so we expected
        the output to change and the recorded `expected` is now stale).

    Either way, this set is what a "retest-all" run would surface, and we
    use it as the GROUND TRUTH for evaluating whether our selection is
    safe (i.e., whether selection missed any test that would have flagged
    a difference).
    """
    failures = []                                  # tids whose output changes on new_func
    for tid, (inp, expected) in enumerate(test_suite):  # iterate over every (input, golden) pair
        actual = new_func(*inp)                    # run the NEW (post-change) function
        if actual != expected:                     # output differs -> this is a real regression hit
            failures.append(tid)                   # record the failing test id
    return failures


def report(name, test_suite, test_coverages, changed_lines, new_func):
    selected = select_tests(test_coverages, changed_lines)
    failures = regression_failures(test_suite, new_func)
    safe = set(failures).issubset(set(selected))
    print(f"--- {name} ---")
    print(f"  Changed lines           : {sorted(changed_lines)}")
    print(f"  Total tests             : {len(test_suite)}")
    print(f"  Selected tests          : {len(selected)} "
          f"({len(selected)/len(test_suite)*100:.1f}%)")
    print(f"  Actual regression fails : {len(failures)}")
    print(f"  Safe selection?         : {safe}  "
          f"(all failing tests are in the selected set)")
    return selected, failures


### Scenario A — threshold change

This is a **condition** mutation: the `if` line itself changed. Every test passes through the `if` check, so every test's coverage includes the changed line.

In [50]:
selected_A, failures_A = report(
    "Scenario A (threshold 40000 -> 50000)",
    test_suite, test_coverages, changed_lines_A, calculate_shipping_fee_A,
)

--- Scenario A (threshold 40000 -> 50000) ---
  Changed lines           : [13]
  Total tests             : 512
  Selected tests          : 512 (100.0%)
  Actual regression fails : 122
  Safe selection?         : True  (all failing tests are in the selected set)


Notice that Scenario A selects **every** test — no reduction.

That's a real (and instructive) limitation of statement-coverage-based selection: changes that live in a condition expression are reachable by every test that evaluates the condition, regardless of which branch the test actually took. **Branch** or **condition** coverage would give a finer signal here.

### Scenario B — branch-body changes

The two changed lines are inside `if` bodies (`membership == "WOW"` branch and `coupon_type == "NEW_USER"` branch). Tests that never take those branches don't cover the changed lines and are safely skipped.

In [51]:
selected_B, failures_B = report(
    "Scenario B (WOW //2 -> //3  AND  NEW_USER -=2000 -> -=3000)",
    test_suite, test_coverages, changed_lines_B, calculate_shipping_fee_B,
)

--- Scenario B (WOW //2 -> //3  AND  NEW_USER -=2000 -> -=3000) ---
  Changed lines           : [38, 42]
  Total tests             : 512
  Selected tests          : 384 (75.0%)
  Actual regression fails : 318
  Safe selection?         : True  (all failing tests are in the selected set)


## Summary

In [52]:
from tabulate import tabulate

rows = [
    ["Scenario A (condition)",
     sorted(changed_lines_A),
     len(test_suite),
     f"{len(selected_A)} ({len(selected_A)/len(test_suite)*100:.1f}%)",
     len(failures_A),
     set(failures_A).issubset(set(selected_A))],
    ["Scenario B (branch bodies)",
     sorted(changed_lines_B),
     len(test_suite),
     f"{len(selected_B)} ({len(selected_B)/len(test_suite)*100:.1f}%)",
     len(failures_B),
     set(failures_B).issubset(set(selected_B))],
]
print(tabulate(
    rows,
    headers=["Scenario", "Changed lines", "|T|", "|T_selected|",
             "Regression fails", "Safe?"],
    tablefmt="github",
))

| Scenario                   | Changed lines   |   |T| | |T_selected|   |   Regression fails | Safe?   |
|----------------------------|-----------------|-------|----------------|--------------------|---------|
| Scenario A (condition)     | [13]            |   512 | 512 (100.0%)   |                122 | True    |
| Scenario B (branch bodies) | [38, 42]        |   512 | 384 (75.0%)    |                318 | True    |


## Reflection

- **Safe but not minimal.** The selected set contains every test that *might* fail and usually some that won't (tests whose coverage includes the changed line but whose actual values don't trigger a behavioral difference). Safety is preserved at the cost of running some extra tests.
- **Coverage granularity matters.** Scenario A exposes a limitation of statement coverage: a condition-only change forces a full re-run. Branch or MCDC coverage would let us narrow the selection (e.g., only tests whose `order_total` is in the new "now-free" zone `[40000, 50000)`).
- **What about added or removed lines?** A line-by-line diff stops working when scenarios add new branches or delete code. Real tools use AST diffs or per-statement IDs, and add downstream dataflow / dependency analysis to remain safe.
- **Compose with prioritization.** Selection chooses *which* tests; prioritization chooses the *order*. In CI you typically combine both: run the selected subset first, ordered by recent failure rate or affected-code distance.